In [2]:
#| default_exp meta_learning.environments.pricing_env.pricing_env

In [3]:
#| export
import gym
from abc import ABC, abstractmethod
from typing import Union, List, Dict
import numpy as np


In [ ]:
#| export
import gym
import numpy as np
from typing import List, Optional


class PricingEnv(gym.Env):
    """
    ──────────────────────────────────────────────────────────────────────────
    Single-SKU pricing environment for meta-RL (RL², variBAD, etc.)
    ──────────────────────────────────────────────────────────────────────────

    Task vector  (length = 2 · F + 3)  
        [ alpha ₀,…,alpha _{F-1},
          beta ₀,…,beta _{F-1},
          sigma,
          inv_ratio,
          horizon ]

      • *alpha* (F)    – intercept terms of the linear demand model  
      • *beta*  (F)    – slope terms (negative ⇒ demand ↓ as price ↑)  
      • *sigma*        – st.dev. of additive Gaussian demand noise  
      • *inv_ratio*    – initial inventory / horizon (0 … 1)  
      • *horizon*      – episode length (integer)

    Observation  
        [ inventory , feature₀ … feature_{F-1} ]

    Action  
        scalar price in ⁠[ p_low , p_high ].

    Reward  
        revenue = price × sales  (no holding- or stock-out costs).

    Notes for meta-RL wrappers
    --------------------------
    • **`reset_task()`** changes *only* the latent task; it does *not* start
      an episode. Call `reset()` afterwards.  
    • `info["task"]` returns the current task vector for logging /
      evaluation.  
    • Boundaries of `task_space` are set to large finite values only to keep
      Gym quiet; the code never samples from that space.

    Reproducibility
    ---------------
    This minimal version uses the **global NumPy RNG** (`np.random`).  If you
    need deterministic roll-outs in a vectorised setup, create your own
    wrapper that seeds the RNG per worker.
    """

    _BIG = 1e9  # very wide bound for observation & task spaces

    # --------------------------------------------------------------------- #
    #                         constructor                                    #
    # --------------------------------------------------------------------- #
    def __init__(self,
                 # price limits
                 p_low: float = 0.0,
                 p_high: float = 1.0,

                 # feature & task-distribution settings
                 nb_features: int = 1,
                 horizon_choices: List[int] = (500,),
                 mean_alpha: float = 1.2,
                 std_alpha: float = 0.0,
                 mean_beta: float = -0.3,
                 std_beta: float = 0.0,
                 noise_std_choices: List[float] = (0.2,),
                 inv_ratio_mean: float = 0.5,
                 inv_ratio_std: float = 0.1,

                 # optional fixed task (otherwise sampled)
                 task: Optional[np.ndarray] = None):

        super().__init__()

        # -------- store hyper-parameters -------------------------------------
        self.nb_features       = nb_features
        self.horizon_choices   = tuple(horizon_choices)
        self.mean_alpha        = mean_alpha
        self.std_alpha         = std_alpha
        self.mean_beta         = mean_beta
        self.std_beta          = std_beta
        self.noise_std_choices = tuple(noise_std_choices)
        self.inv_ratio_mean    = inv_ratio_mean
        self.inv_ratio_std     = inv_ratio_std

        # -------- Gym spaces --------------------------------------------------
        # action: scalar price -------------------------------------------------
        self.action_space = gym.spaces.Box(
            low=np.array([p_low],  dtype=np.float32),
            high=np.array([p_high], dtype=np.float32),
            dtype=np.float32
        )

        # observation: inventory + features -----------------------------------
        obs_dim = 1 + nb_features
        self.observation_space = gym.spaces.Box(
            low=-self._BIG, high=self._BIG,
            shape=(obs_dim,), dtype=np.float32
        )

        # task vector (never actually sampled by the library code) ------------
        self.task_dim = 2 * nb_features + 3
        self.task_space = gym.spaces.Box(
            low=-self._BIG, high=self._BIG,
            shape=(self.task_dim,), dtype=np.float32
        )

        # -------- set latent task & episode ----------------------------------
        self.reset_task(task)

    # ===================================================================== #
    #                      meta-learning hooks                               #
    # ===================================================================== #
    def reset_task(self, task: Optional[np.ndarray] = None) -> None:
        """
        Pick a new latent task.

        Parameters
        ----------
        task : ndarray or None
            • *None*  – sample from the task prior  
            • ndarray – use given vector (length == `task_dim`)
        """
        if task is None:
            task = self._sample_task()
        task = np.asarray(task, dtype=np.float32)
        assert task.shape == (self.task_dim,), f"task has wrong shape {task.shape}"
        self._task = task

        F = self.nb_features
        self.alpha     = task[:F]
        self.beta      = task[F:2*F]
        self.sigma     = float(task[2*F])
        self.inv_ratio = float(np.clip(task[2*F + 1], 0.00, 1.0))
        self.horizon   = int(task[2*F + 2])

    def get_task(self) -> np.ndarray:
        """Return **copy** of the current task vector (float32)."""
        return self._task.copy()

    # ===================================================================== #
    #                       Gym interface                                   #
    # ===================================================================== #
    def reset(self):
        """Start a fresh episode under the *current* task."""
        self.t   = 0
        self.inv = float(self.horizon * self.inv_ratio)
        return self._get_obs()

    def step(self, action):
        # ── 1. clip price into valid range ---------------------------------------
        price = float(np.clip(action, self.action_space.low[0],
                                    self.action_space.high[0]))

        # ── 2. realise (stochastic) demand ---------------------------------------
        noise   = np.random.normal(0.0, self.sigma)
        demand  = self._demand(price, noise)

        # ── 3. apply / skip inventory constraint ---------------------------------
        if self.inv_ratio == 0.0:                # ▶ unlimited stock
            sales = demand                       #   no inventory depletion
        else:                                    # ▶ finite stock
            sales = min(demand, self.inv)
            self.inv -= sales

        # ── 4. reward (revenue) ---------------------------------------------------
        reward = price * sales

        # ── 5. book-keeping -------------------------------------------------------
        self.t += 1
        done = (self.t >= self.horizon) or (self.inv <= 0.0)

        obs = self._get_obs()
        info = {
            "task":   self.get_task(),
            "noise":  noise,
            "demand": demand,
            "sales":  sales,
            "inv":    self.inv
        }
        return obs, reward, done, info


    # ===================================================================== #
    #                         internal helpers                               #
    # ===================================================================== #
    # ---------- task sampler --------------------------------------------------
    def _sample_task(self) -> np.ndarray:
        F = self.nb_features
        alpha = np.random.normal(self.mean_alpha, self.std_alpha,  size=F)
        beta  = np.random.normal(self.mean_beta,  self.std_beta,   size=F)
        sigma = float(np.random.choice(self.noise_std_choices))
        inv_ratio = float(np.clip(
            np.random.normal(self.inv_ratio_mean, self.inv_ratio_std), 0.00, 1.0))
        horizon = int(np.random.choice(self.horizon_choices))

        return np.concatenate([alpha, beta,
                               [sigma, inv_ratio, horizon]]).astype(np.float32)

    # ---------- observation helpers ------------------------------------------
    def _get_features(self) -> np.ndarray:
        """Draw a single feature vector Xₜ."""
        if self.nb_features == 1:
            return np.ones(1, dtype=np.float32)
        scale = 1.0 / np.sqrt(self.nb_features - 1)
        return np.random.uniform(0.0, scale, size=self.nb_features).astype(np.float32)

    def _get_obs(self) -> np.ndarray:
        self._X = self._get_features()
        return np.concatenate([[self.inv], self._X]).astype(np.float32)

    # ---------- demand model --------------------------------------------------
    def _demand(self, price: float, noise: float) -> float:
        """Linear demand with additive noise; demand ≥ 0."""
        mean = float(np.dot(self.alpha - self.beta * price, self._X))
        return max(0.0, mean + noise)

    # ---------- visualisation stub -------------------------------------------
    def visualise_behaviour(self, *_, **__):
        """
        Optional — leave blank.  Hyper’s default visualiser is used if None.
        """
        return None, None, None, None, None, None, None


In [235]:
# Create a test environment
test_env = PricingEnv(p_low=0, p_high=5, 
                      nb_features=5, horizon_choices=[500],
                      mean_alpha=1.2, std_alpha=0.2,
                      mean_beta=-0.3, std_beta=0.2,
                      noise_std_choices=[0.2], 
                      inv_ratio_mean=0.0, 
                      inv_ratio_std=0.0)

In [ ]:
# Reset the task
test_env.reset_task()

In [233]:
test_env.get_task()

array([ 1.2808509e+00,  1.2511061e+00,  1.3539151e+00,  1.3662823e+00,
        1.0312239e+00, -2.0335504e-01, -2.4037448e-01, -4.2371634e-02,
       -2.5720444e-01,  2.1198297e-01,  2.0000000e-01,  9.9999998e-03,
        5.0000000e+02], dtype=float32)

In [234]:
test_env.reset()

array([5.        , 0.24304228, 0.31387925, 0.39711162, 0.13783522,
       0.31750578], dtype=float32)

In [229]:
action = np.array([2])  # Example action within the bounds
test_env.step(action)

(array([2.5561323 , 0.14113972, 0.38865873, 0.21106511, 0.02532968,
        0.29373744], dtype=float32),
 4.8877350978414045,
 False,
 {'task': array([ 1.0200435e+00,  1.2350589e+00,  1.3257301e+00,  1.1770791e+00,
          7.3284781e-01, -4.9770367e-01, -1.7388670e-01, -4.6250677e-01,
         -3.6380973e-01, -3.6084002e-01,  2.0000000e-01,  9.9999998e-03,
          5.0000000e+02], dtype=float32),
  'noise': -0.15097393133442455,
  'demand': 2.4438675489207022,
  'sales': 2.4438675489207022,
  'inv': 2.556132339320589})